import pandas as pd
import os
import json

In [13]:
import pandas as pd
import os
import json

output_dir = os.path.join('..','Database', 'processed')
imputed_filename = 'imputedv1.csv'

imputed = pd.read_csv(os.path.join(output_dir, imputed_filename))

column_dict_filepath = os.path.join('..','Database','processed','form_1_column_dict.json')

with open(column_dict_filepath, 'r') as f:
    column_dict = json.load(f)

categorical_all = column_dict['ordinal_col'] + column_dict['nominal_col']

for col in categorical_all:
    if col in imputed.columns:
        imputed[col] = imputed[col].round().astype(int)

imputed = imputed.drop(columns=['Unnamed: 0','Mod1Id'])

In [14]:
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

ord_cols = [col for col in column_dict['ordinal_col'] if col in imputed.columns]
cat_cols = [col for col in column_dict['nominal_col'] if col in imputed.columns]
num_cols = [col for col in column_dict['continuous_col'] if col in imputed.columns]

ord_encoder = OrdinalEncoder()
scalar = StandardScaler()

imputed[ord_cols] = ord_encoder.fit_transform(imputed[ord_cols])
imputed[num_cols] = scalar.fit_transform(imputed[num_cols])


imputed[num_cols] = imputed[num_cols].astype(float)
imputed[cat_cols] = imputed[cat_cols].astype('category')

for col in ord_cols:
    imputed[col] = pd.Categorical(imputed[col], ordered=True)

print(imputed.shape)
imputed.head()

(19560, 223)


,cntAnyAfterIndex,cntAnyBefore15yr,cntAnyBeforeIndex,cntAnyInjuries,cntAnySameIndex,cntLOCAfterIndex,cntLOCBefore15yr,cntLOCBeforeIndex,cntLOCInjuries,cntLOCSameIndex,...,SCI,SexF,SmkCig,SpEd,Suicide,TBI_IDAsked,WordRecallTCC,AgeGroup,InjuryPeriod,ZipInj
0,-0.345079,-0.207726,-0.378453,-0.497003,-0.113050,-0.287624,-0.085034,-0.256641,-0.360243,-0.012630,...,0,1,0.0,0,0,1,7,2.0,1,9
1,-0.511839,-0.755086,-0.373600,-0.685569,-0.884285,0.243748,-0.593528,-0.256662,-0.334779,-1.845412,...,0,2,2.0,0,0,0,9,2.0,1,9
2,4.403516,-1.963656,-2.540299,0.064332,-0.871442,2.480581,0.138432,-0.256660,1.082122,-1.336273,...,0,2,0.0,0,0,0,11,0.0,2,43055
3,-0.345079,-0.207726,-0.378453,-0.497003,-0.113050,-0.287624,-0.085034,-0.256641,-0.360243,-0.012630,...,0,1,0.0,0,0,1,7,8.0,3,10024
4,0.814036,-1.212343,-1.208557,-0.774802,-0.239410,-0.118161,-0.126232,-0.256660,-0.275594,0.155644,...,0,2,0.0,0,0,0,7,8.0,1,9


In [15]:
from prince import FAMD

famd = FAMD(n_components=2)
famd = famd.fit(imputed)

# Transform data
df_famd = famd.transform(imputed)
df_famd.columns = ['FAMD_1', 'FAMD_2']

c:\Users\benol\miniconda3\envs\ps\Lib\site-packages\prince\famd.py:86: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  eta2[col] = (
c:\Users\benol\miniconda3\envs\ps\Lib\site-packages\prince\famd.py:86: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  eta2[col] = (
c:\Users\benol\miniconda3\envs\ps\Lib\site-packages\prince\famd.py:86: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(a

In [17]:
import altair as alt

#essential for altier to work. very finicky 
alt.data_transformers.disable_max_rows()
alt.data_transformers.enable('default', max_rows=None)

# Optional: add labels from original df
df_famd['label'] = imputed['FIMLocoModeD'].dropna()

# Altair plot
chart = alt.Chart(df_famd.reset_index()).mark_circle().encode(
    x='FAMD_1',
    y='FAMD_2',
    color='label:N',
    tooltip='label'
).interactive()

chart

alt.Chart(...)

In [18]:
print("Top contributors to FAMD 01:")
famd.column_contributions_.sort_values(by=0,ascending=False).head(10).style.format('{:.3%}')

Top contributors to FAMD 01:


component,0,1
variable,,
FIMTOTD,0.031%,0.000%
FIMMOTD,0.027%,0.001%
FIMTOTA,0.027%,0.006%
FIMMOTA,0.024%,0.009%
DRSd,0.024%,0.003%
DRSdHigh,0.023%,0.004%
DRSdLow,0.023%,0.003%
FIMBedTransD,0.022%,0.039%
FIMToilTransD,0.022%,0.041%


In [19]:
print("Top contributors to FAMD 02:")
top_famd_2 = famd.column_contributions_.sort_values(by=1,ascending=False).head(10).style.format('{:.3%}')
top_famd_2

Top contributors to FAMD 02:


component,0,1
variable,,
FIMToilTransD,0.022%,0.041%
FIMBedTransD,0.022%,0.039%
FIMDrsdwnD,0.022%,0.038%
FIMToiletD,0.022%,0.037%
FIMDrupD,0.021%,0.036%
FIMBathD,0.021%,0.036%
FIMGroomD,0.022%,0.034%
ZipInj,0.013%,0.033%
FIMLocoD,0.018%,0.030%


In [21]:
import hdbscan

clusterer = hdbscan.HDBSCAN(min_cluster_size=10)

df_famd['cluster'] = clusterer.fit_predict(df_famd[['FAMD_1','FAMD_2']])

df_famd_denoised = df_famd[df_famd['cluster'] != -1]

df_famd['label'] = imputed['FIMLocoModeD']

# Altair plot
chart = alt.Chart(df_famd_denoised.reset_index(), title="FAMD Components: FIMLocoModeD").mark_circle().encode(
    x='FAMD_1',
    y='FAMD_2',
    color='label:N',
    tooltip='label'
).properties(
    width=800,   # increase width
    height=600   # increase height
).interactive()

chart

c:\Users\benol\miniconda3\envs\ps\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\benol\miniconda3\envs\ps\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


alt.Chart(...)